In [1]:
import polars as pl
import polars.selectors as cs
import soccerdata as sd
import matplotlib.pyplot as plt
import sklearn as skl
import pandas as pd
import numpy as np
import pyarrow
import seaborn as sb
import bs4
import plotly.express as px
import math
import json
import os
import requests
from fontTools.misc.cython import returns
import socceraction.spadl as spadl
from mplsoccer import Pitch, FontManager, Sbopen, VerticalPitch
import pandas.errors
from pathlib import Path

[03/04/26 11:52:12] INFO     No custom team name replacements found. You can configure these in       ]8;id=290406;file://C:\Users\yukiw\PycharmProjects\pythonProject\.venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=532300;file://C:\Users\yukiw\PycharmProjects\pythonProject\.venv\Lib\site-packages\soccerdata\_config.py#92\92]8;;\
                             C:\Users\yukiw\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=111580;file://C:\Users\yukiw\PycharmProjects\pythonProject\.venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=378682;file://C:\Users\yukiw\PycharmProjects\pythonProject\.venv\Lib\site-packages\soccerdata\_config.py#198\198]8;;\
                             C:\Users\yukiw\soccerdata\config\league_dict.json.                                    

In [2]:
from pyfonts import load_google_font
font = load_google_font("DotGothic16")

In [12]:
"""
def splitter(df):

    df['x'] *= 1.2
    df['end_x'] *= 1.2
    df['y'] *= 0.8
    df['end_y'] *= 0.8
    df.insert(1, "pass_angle",
    np.degrees(np.arctan2(
        df['end_y'] - df['y'],
        df['end_x'] - df['x']
        ))
    )
    df.insert(3, 'receiver',
        df['player'].shift(-1)
    )
    return df
    """

In [3]:
def find_games(team):
    path = Path(f'C:/Users/yukiw/Documents/rp/footy_analysis/testers/teams/{team}')
    games = []
    for file in path.iterdir():
        try:
            games.append(pd.read_csv(str(file)))
        except pd.errors.EmptyDataError:
            continue
    return games

In [4]:
def analyze_games(team, player):
    all_game_result = []
    all_game_time_result = []
    path = Path(f'C:/Users/yukiw/Documents/rp/footy_analysis/testers/teams/{team}')
    for file in path.iterdir():
        try:
            df = pd.read_csv(str(file))
            df['x'] *= 1.2
            df['end_x'] *= 1.2
            df['y'] *= 0.8
            df['end_y'] *= 0.8
            df.insert(1, "pass_angle",
            np.degrees(np.arctan2(
                df['end_y'] - df['y'],
                df['end_x'] - df['x']
                ))
            )
            df.insert(3, 'receiver',
                df['player'].shift(-1)
            )
            
            target_player = player
            cas = df[df['player'] == target_player]
            ids = cas['player_id']
            player_received = df[df['receiver'] == target_player]
            player_performance = df[df['player'] == target_player]
            player_received = player_received.loc[player_received['outcome_type'] == 'Successful']
            player_received = player_received.loc[player_received['team'] == team]
            actions = [player_received,player_performance]
            all_actions = pd.concat(actions)
            all_actions = all_actions.sort_values(by=['minute', 'second'])
            result = []
            time_list = []
            for i in range(len(all_actions) - 1):
                current = all_actions.iloc[i]
                next_row = all_actions.iloc[i + 1]
                if current['player'] != player and next_row['player'] == player:
                    if next_row['type'] in ['Pass','TakeOn', 'BallTouch', 'Clearance']:
                        result.append(next_row['outcome_type'])
                        receive_time = current['minute'] * 60 + current['second']
                        action_time  = next_row['minute'] * 60 + next_row['second']
                        time_list.append(action_time - receive_time)
            all_game_result.append(result)
            all_game_time_result.append(time_list)
        except:
            continue
            
    return all_game_result, all_game_time_result

In [51]:
import statistics as stat
def find_accuracy(results):
    accuracy_list = []
    for i in range(len(results)):
        if len(results[i]) > 0:
            tester = [0 if i == 'Unsuccessful' else 1 for i in results[i]]
            accuracy_list.append(stat.mean(tester))
        else:
            pass
        i+=1
    return stat.mean(accuracy_list), stat.variance(accuracy_list)

In [9]:
simons_spurs_result, simons_spurs_time_result = analyze_games('Tottenham', 'Xavi Simons')

In [6]:
simons_leipzig_result, simons_leipzig_time_result = analyze_games('RBL', 'Xavi Simons')

In [55]:
simons_spurs_rate, variance = find_accuracy(simons_spurs_result)
#simons_spurs_time =
simons_leigzig_rate, variance2 = find_accuracy(simons_leipzig_result)
#simons_leipzig_time

In [53]:
variance

0.009258681596403539

In [56]:
variance2

0.004686100136829372

In [207]:
def collect_teams(league, season,team):
    path = f"C:\\Users\\yukiw\\Documents\\rp\\footy_analysis\\testers\\teams\\{team}" 
    os.mkdir(path)
    ws = sd.WhoScored(leagues=league, seasons=season)
    league = ws.read_schedule()
    home = league[league['home_team'] == team]
    away = league[league['away_team'] == team]
    t = pd.concat([home, away])
    list = t['game_id'].to_list()        
    t_list =  []
    for i in range(len(list)):
        game = ws.read_events(match_id = list[i])
        game.to_csv(path + "\\game_" + str(i) + ".csv", index=False)
        t_list.append(game)
    return t_list

In [ ]:
collect_teams(league="GER-Bundesliga", season=2024, team="Leverkusen")

In [ ]:
collect_teams(league="GER-Bundesliga", season=2024, team="RBL")